[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/42.%20The%20Peak%20Energy%20Consumption%20%26%20Load%20Shifting%20Problem/P42-Tier-5_executed.ipynb)

---

# 42. The Peak Energy Consumption and Load-Shifting Problem

## Tier 5: Integrated Digital-Twin Simulation

### Goal

Build an illustrative digital-twin simulation that combines models of an electrical grid, thermal systems, an equipment fleet, weather, and battery storage. The notebook synchronizes the virtual state with supplied physical observations, generates subsystem forecasts, aggregates demand and price forecasts, and presents selected system indicators.

This tier emphasizes integrated monitoring and prediction. It does not solve an optimization problem or automatically control real equipment.

### Inputs and outputs

The demonstration supplies a small dictionary of synthetic physical observations, including grid demand and ambient temperature. Subsystem models also use seeded random variation to represent uncertain operating conditions. The outputs are a virtual system state, subsystem updates, forecasts, and visual summaries. Results depend on the simulation assumptions and are not measurements from a real facility.

### Model assumptions and scope

- Electrical demand and grid capacity are measured in MW; equipment and thermal power are measured in kW; stored energy is measured in kWh; and electricity prices are measured in dollars per kWh.
- Each synchronization call represents a 15-minute simulation step for storage. The subsystem update frequencies describe intended update intervals; this notebook does not run a real-time clock or connect to physical devices.
- The supplied grid demand is the observed grid load used by the electrical-grid model. The virtual state's `current_demand` is instead the modeled facility load from thermal, equipment, and storage subsystems. The two quantities have distinct roles and are not added together.
- Thermal and equipment forecasts are simplified scenario models. Weather affects the thermal contribution to aggregated demand through a load factor. Forecast uncertainty bands in the visualization are illustrative bounds, not statistically calibrated confidence intervals.
- The reported synchronization score is a model-confidence proxy formed from subsystem `accuracy_score` values. Without measured physical-versus-virtual residuals, it is not an empirical synchronization-accuracy estimate.
- Grid dynamics, network constraints, equipment interactions, battery degradation, and market prices are represented only at a simplified level. No scenario-based resilience study or predictive optimization is performed here.

### Integration logic

For a synchronization step, the virtual flexible-load estimate is

$$
D_t^{\mathrm{virtual}} = \frac{F_t + W_t^{\mathrm{thermal}} T_t}{1000} + \frac{P_t^{\mathrm{storage}}}{1000},
$$

where $F_t$ is thermal-system power in kW, $W_t^{\mathrm{thermal}}$ is the weather-related thermal load factor, $T_t$ is equipment-fleet power in kW, and $P_t^{\mathrm{storage}}$ is storage power in kW. Positive storage power means charging, which increases demand; negative power means discharging, which reduces it. In the implementation, `IntegratedDigitalTwin._update_aggregated_state` applies the weather factor to thermal load, adds equipment load, and then adds signed storage power. The grid model separately uses observed `total_demand` in MW.

For each forecast hour $h$, the aggregated modeled demand is

$$
\widehat{D}_{t+h} = \frac{\widehat{F}_{t+h} + \widehat{T}_{t+h}}{1000} + \frac{\widehat{P}_{t+h}^{\mathrm{storage}}}{1000},
$$

where the hatted quantities are the thermal, fleet, and storage forecasts in their respective kW units. The code uses the requested forecast horizon and represents storage charging as positive demand and discharging as negative demand.

### What to examine

- How subsystem observations update the shared virtual state.
- How the grid, thermal, equipment, weather, and storage models contribute different state variables and forecasts.
- Why power values must be converted consistently before they are aggregated.
- How forecast values and operational indicators should be interpreted in light of the simplified assumptions.

In [ ]:
from collections import defaultdict, deque
from copy import deepcopy
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import random
import time
from typing import Any, Dict, List, Optional, Tuple
import warnings

import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore")

# Fixed seeds make the illustrative random variations reproducible.
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")

In [ ]:
@dataclass
class DigitalTwinState:
    """Shared virtual state for the integrated energy-system model."""

    timestamp: datetime
    physical_sync_accuracy: float = 1.0
    virtual_sync_accuracy: float = 1.0
    system_health_score: float = 1.0
    current_demand: float = 0.0  # Modeled flexible facility load, MW
    current_supply: float = 0.0  # MW
    current_price: float = 0.15  # dollars per kWh
    storage_soc: float = 0.5
    equipment_efficiency: float = 0.85
    thermal_comfort_score: float = 0.9
    grid_stability_score: float = 0.95
    demand_forecast_1h: float = 0.0
    demand_forecast_4h: float = 0.0
    demand_forecast_24h: float = 0.0
    price_forecast_1h: float = 0.0
    price_forecast_4h: float = 0.0
    price_forecast_24h: float = 0.0


@dataclass
class SubsystemModel:
    """Base interface shared by the subsystem models."""

    name: str
    model_type: str
    update_frequency: int  # Intended update interval, seconds
    last_update: datetime = field(default_factory=datetime.now)
    health_status: str = "healthy"
    accuracy_score: float = 1.0

    def update_state(
        self,
        physical_data: Dict[str, Any],
        virtual_state: DigitalTwinState,
    ) -> Dict[str, Any]:
        """Update subsystem state using observations and the virtual state."""
        raise NotImplementedError

    def predict_behavior(
        self,
        horizon_hours: int,
        scenario_data: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Return a forecast for the requested number of hours."""
        raise NotImplementedError


@dataclass
class ElectricalGridModel(SubsystemModel):
    """Simplified electrical-grid model with hourly price patterns."""

    def __init__(self):
        super().__init__("Electrical Grid", "Power System", 30)
        self.base_voltage = 13.8  # kV
        self.grid_frequency = 50.0  # Hz
        self.max_capacity = 50.0  # MW
        self.grid_stability_margin = 0.15
        self.base_price_profile = self._generate_base_price_profile()
        self.price_volatility = 0.2
        self.current_load = 0.0  # MW
        self.voltage_regulation_active = True
        self.frequency_deviation = 0.0

    def _generate_base_price_profile(self) -> Dict[int, float]:
        """Create a seeded, illustrative price profile for hours 1 through 24."""
        prices = {}
        for hour in range(1, 25):
            if 6 <= hour <= 10:
                prices[hour] = 0.18 + random.uniform(-0.02, 0.02)
            elif 18 <= hour <= 22:
                prices[hour] = 0.20 + random.uniform(-0.02, 0.02)
            elif hour == 24 or hour <= 5:
                prices[hour] = 0.08 + random.uniform(-0.01, 0.01)
            else:
                prices[hour] = 0.12 + random.uniform(-0.01, 0.01)
        return prices

    def update_state(
        self,
        physical_data: Dict[str, Any],
        virtual_state: DigitalTwinState,
    ) -> Dict[str, Any]:
        """Update grid indicators from the observed grid load."""
        current_hour = virtual_state.timestamp.hour or 24
        self.current_load = float(
            physical_data.get("total_demand", virtual_state.current_demand)
        )
        load_ratio = self.current_load / self.max_capacity

        if load_ratio > 1.0 - self.grid_stability_margin:
            self.frequency_deviation = (
                load_ratio - (1.0 - self.grid_stability_margin)
            ) * 2.0
            self.health_status = "stressed"
        else:
            self.frequency_deviation = 0.0
            self.health_status = "healthy"

        base_price = self.base_price_profile[current_hour]
        current_price = base_price * random.uniform(
            1.0 - self.price_volatility,
            1.0 + self.price_volatility,
        )
        if load_ratio > 0.8:
            current_price *= 1.2

        self.last_update = virtual_state.timestamp
        return {
            "current_price": current_price,
            "grid_load": self.current_load,
            "load_ratio": load_ratio,
            "frequency_deviation": self.frequency_deviation,
            "health_status": self.health_status,
            "voltage_regulation_active": self.voltage_regulation_active,
        }

    def predict_behavior(
        self,
        horizon_hours: int,
        scenario_data: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Forecast hourly prices, loads, and stability indicators."""
        predictions = {
            "price_forecast": [],
            "load_forecast": [],
            "stability_forecast": [],
        }
        current_time = scenario_data.get("current_time", datetime.now())
        base_demand = float(scenario_data.get("base_demand", self.current_load))

        for hour_offset in range(1, horizon_hours + 1):
            future_hour = (current_time + timedelta(hours=hour_offset)).hour or 24
            base_price = self.base_price_profile[future_hour]
            demand_factor = 1.0 + (base_demand / self.max_capacity - 0.5) * 0.3
            predictions["price_forecast"].append(base_price * demand_factor)

            if 6 <= future_hour <= 10 or 18 <= future_hour <= 22:
                load_multiplier = 1.2
            elif future_hour <= 5:
                load_multiplier = 0.7
            else:
                load_multiplier = 1.0

            predicted_load = (
                base_demand * load_multiplier * random.uniform(0.9, 1.1)
            )
            predictions["load_forecast"].append(predicted_load)
            load_ratio = predicted_load / self.max_capacity
            stability = max(0.0, 1.0 - max(0.0, load_ratio - 0.85) * 5.0)
            predictions["stability_forecast"].append(stability)

        return predictions


electrical_grid = ElectricalGridModel()

In [ ]:
@dataclass
class ThermalSystemModel(SubsystemModel):
    """Simplified HVAC and refrigeration model; power values are in kW."""

    def __init__(self):
        super().__init__("Thermal System", "HVAC & Refrigeration", 60)
        self.ambient_temperature = 22.0
        self.target_temperature = 20.0
        self.temperature_tolerance = 1.0
        self.hvac_units = {
            "hvac_1": {"capacity": 500, "current_load": 0.0, "efficiency": 0.85, "status": "auto"},
            "hvac_2": {"capacity": 500, "current_load": 0.0, "efficiency": 0.87, "status": "auto"},
            "hvac_3": {"capacity": 300, "current_load": 0.0, "efficiency": 0.83, "status": "auto"},
        }
        self.refrigeration_units = {
            "cold_storage_1": {"capacity": 200, "current_load": 0.0, "efficiency": 0.75, "setpoint": -18.0},
            "cold_storage_2": {"capacity": 150, "current_load": 0.0, "efficiency": 0.78, "setpoint": -15.0},
            "freezer_1": {"capacity": 100, "current_load": 0.0, "efficiency": 0.72, "setpoint": -25.0},
        }
        self.comfort_score = 0.9
        self.energy_consumption = 0.0

    def update_state(
        self,
        physical_data: Dict[str, Any],
        virtual_state: DigitalTwinState,
    ) -> Dict[str, Any]:
        """Estimate thermal-system power and a simple comfort indicator."""
        self.ambient_temperature = float(
            physical_data.get("ambient_temp", self.ambient_temperature)
        )
        temp_diff = abs(self.ambient_temperature - self.target_temperature)
        hvac_load_factor = min(1.0, temp_diff / 5.0)
        total_hvac_load = 0.0

        for unit_data in self.hvac_units.values():
            if unit_data["status"] == "auto":
                unit_load = unit_data["capacity"] * hvac_load_factor
            elif unit_data["status"] == "on":
                unit_load = unit_data["capacity"]
            else:
                unit_load = 0.0
            unit_data["current_load"] = unit_load
            total_hvac_load += unit_load

        total_refrigeration_load = 0.0
        ambient_factor = max(0.0, 1.0 + (self.ambient_temperature - 20.0) * 0.05)
        for unit_data in self.refrigeration_units.values():
            unit_load = (
                unit_data["capacity"]
                * ambient_factor
                * random.uniform(0.8, 1.2)
            )
            unit_data["current_load"] = min(
                unit_load,
                unit_data["capacity"] * 1.2,
            )
            total_refrigeration_load += unit_data["current_load"]

        self.energy_consumption = total_hvac_load + total_refrigeration_load
        excess_difference = max(0.0, temp_diff - self.temperature_tolerance)
        self.comfort_score = max(0.0, 1.0 - excess_difference / 5.0)
        self.last_update = virtual_state.timestamp

        return {
            "thermal_load": self.energy_consumption,
            "hvac_load": total_hvac_load,
            "refrigeration_load": total_refrigeration_load,
            "ambient_temperature": self.ambient_temperature,
            "comfort_score": self.comfort_score,
            "hvac_efficiency": float(np.mean([
                unit["efficiency"] for unit in self.hvac_units.values()
            ])),
            "refrigeration_efficiency": float(np.mean([
                unit["efficiency"] for unit in self.refrigeration_units.values()
            ])),
        }

    def predict_behavior(
        self,
        horizon_hours: int,
        scenario_data: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Generate a simple temperature-dependent thermal forecast."""
        predictions = {
            "thermal_load_forecast": [],
            "comfort_score_forecast": [],
            "temperature_forecast": [],
        }
        current_temp = self.ambient_temperature
        current_time = scenario_data.get("current_time", datetime.now())

        for hour_offset in range(1, horizon_hours + 1):
            future_hour = (current_time + timedelta(hours=hour_offset)).hour
            if 6 <= future_hour <= 18:
                temp_change = random.uniform(0.5, 2.0)
            else:
                temp_change = random.uniform(-1.0, 0.5)
            predicted_temp = current_temp + temp_change
            predictions["temperature_forecast"].append(predicted_temp)

            temp_diff = abs(predicted_temp - self.target_temperature)
            load_factor = min(1.0, temp_diff / 5.0)
            predictions["thermal_load_forecast"].append(
                self.energy_consumption * (0.8 + 0.4 * load_factor)
            )
            excess_difference = max(0.0, temp_diff - self.temperature_tolerance)
            predictions["comfort_score_forecast"].append(
                max(0.0, 1.0 - excess_difference / 5.0)
            )
            current_temp = predicted_temp

        return predictions


@dataclass
class EquipmentFleetModel(SubsystemModel):
    """Simplified fleet model; equipment power values are in kW."""

    def __init__(self):
        super().__init__("Equipment Fleet", "Material Handling", 30)
        self.equipment = {
            "crane_a": {"type": "crane", "power_rating": 350, "current_power": 0, "status": "idle", "efficiency": 0.85},
            "crane_b": {"type": "crane", "power_rating": 350, "current_power": 0, "status": "idle", "efficiency": 0.87},
            "crane_c": {"type": "crane", "power_rating": 400, "current_power": 0, "status": "idle", "efficiency": 0.83},
            "conveyor_1": {"type": "conveyor", "power_rating": 120, "current_power": 0, "status": "idle", "efficiency": 0.90},
            "conveyor_2": {"type": "conveyor", "power_rating": 100, "current_power": 0, "status": "idle", "efficiency": 0.88},
            "conveyor_3": {"type": "conveyor", "power_rating": 80, "current_power": 0, "status": "idle", "efficiency": 0.91},
            "ev_charger_1": {"type": "ev_charger", "power_rating": 50, "current_power": 0, "status": "idle", "efficiency": 0.92},
            "ev_charger_2": {"type": "ev_charger", "power_rating": 50, "current_power": 0, "status": "idle", "efficiency": 0.94},
            "lighting": {"type": "lighting", "power_rating": 75, "current_power": 75, "status": "always_on", "efficiency": 0.95},
            "security": {"type": "security", "power_rating": 90, "current_power": 90, "status": "always_on", "efficiency": 0.98},
            "monitoring": {"type": "monitoring", "power_rating": 25, "current_power": 25, "status": "always_on", "efficiency": 0.99},
        }
        self.total_power_consumption = 0.0
        self.fleet_efficiency = 0.85
        self.active_equipment_count = 0
        self.operational_schedule = {}
        self.maintenance_schedule = {}

    def update_state(
        self,
        physical_data: Dict[str, Any],
        virtual_state: DigitalTwinState,
    ) -> Dict[str, Any]:
        """Update equipment activity using time-of-day operating patterns."""
        hour = virtual_state.timestamp.hour
        self.active_equipment_count = 0
        self.total_power_consumption = 0.0

        for equipment_data in self.equipment.values():
            equipment_type = equipment_data["type"]
            if equipment_data["status"] == "always_on":
                equipment_data["current_power"] = equipment_data["power_rating"]
            elif equipment_type == "crane" and 8 <= hour <= 18 and random.random() < 0.7:
                equipment_data["current_power"] = (
                    equipment_data["power_rating"] * random.uniform(0.6, 1.0)
                )
                equipment_data["status"] = "active"
            elif equipment_type == "conveyor" and 8 <= hour <= 18 and random.random() < 0.6:
                equipment_data["current_power"] = (
                    equipment_data["power_rating"] * random.uniform(0.5, 1.0)
                )
                equipment_data["status"] = "active"
            elif equipment_type == "ev_charger" and (hour < 8 or hour > 18) and random.random() < 0.4:
                equipment_data["current_power"] = (
                    equipment_data["power_rating"] * random.uniform(0.8, 1.0)
                )
                equipment_data["status"] = "active"
            else:
                equipment_data["current_power"] = 0.0
                equipment_data["status"] = "idle"

            if equipment_data["status"] in ("active", "always_on"):
                self.active_equipment_count += 1
            self.total_power_consumption += equipment_data["current_power"]

        active_efficiencies = [
            item["efficiency"]
            for item in self.equipment.values()
            if item["status"] == "active"
        ]
        self.fleet_efficiency = (
            float(np.mean(active_efficiencies)) if active_efficiencies else 0.85
        )
        self.last_update = virtual_state.timestamp

        return {
            "fleet_power_consumption": self.total_power_consumption,
            "active_equipment_count": self.active_equipment_count,
            "fleet_efficiency": self.fleet_efficiency,
            "equipment_status": {
                name: data["status"] for name, data in self.equipment.items()
            },
            "equipment_utilization": (
                self.active_equipment_count / len(self.equipment)
            ),
        }

    def predict_behavior(
        self,
        horizon_hours: int,
        scenario_data: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Forecast fleet power and activity from the operating-hour pattern."""
        predictions = {
            "power_consumption_forecast": [],
            "active_equipment_forecast": [],
            "efficiency_forecast": [],
        }
        current_time = scenario_data.get("current_time", datetime.now())
        operational_count = sum(
            item["type"] in ("crane", "conveyor")
            for item in self.equipment.values()
        )
        essential_count = sum(
            item["status"] == "always_on" for item in self.equipment.values()
        )

        for hour_offset in range(1, horizon_hours + 1):
            hour = (current_time + timedelta(hours=hour_offset)).hour
            if 8 <= hour <= 18:
                predicted_power = self.total_power_consumption * random.uniform(0.8, 1.2)
                predicted_active = operational_count * 0.7
            else:
                predicted_power = self.total_power_consumption * 0.3
                predicted_active = essential_count
            predictions["power_consumption_forecast"].append(predicted_power)
            predictions["active_equipment_forecast"].append(predicted_active)
            predictions["efficiency_forecast"].append(
                self.fleet_efficiency * random.uniform(0.95, 1.05)
            )

        return predictions


thermal_system = ThermalSystemModel()
equipment_fleet = EquipmentFleetModel()

In [ ]:
@dataclass
class WeatherSystemModel(SubsystemModel):
    """Illustrative weather model for environmental conditions."""

    def __init__(self):
        super().__init__("Weather System", "Environmental", 300)
        self.current_temperature = 22.0
        self.current_humidity = 0.6
        self.wind_speed = 5.0
        self.solar_irradiance = 800.0
        self.cloud_cover = 0.3
        self.forecast_horizon = 72
        self.weather_patterns = self._initialize_weather_patterns()
        self.season = "summer"
        self.seasonal_adjustment = 1.0

    def _initialize_weather_patterns(self) -> Dict[str, Any]:
        """Return the daily patterns used by the simplified weather model."""
        return {
            "daily_temp_range": 8.0,
            "humidity_range": 0.3,
            "wind_pattern": [5.0, 6.0, 7.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 4.0, 5.0, 6.0],
            "solar_pattern": [0, 0, 0, 50, 200, 400, 600, 800, 900, 800, 600, 400, 200, 50, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            "weather_events": [],
        }

    def update_state(
        self,
        physical_data: Dict[str, Any],
        virtual_state: DigitalTwinState,
    ) -> Dict[str, Any]:
        """Update weather indicators using a seasonal daily pattern."""
        hour = virtual_state.timestamp.hour
        base_temp = {"winter": 5.0, "spring": 15.0, "fall": 18.0}.get(
            self.season, 22.0
        )
        temp_variation = self.weather_patterns["daily_temp_range"] * np.sin(
            2 * np.pi * (hour - 6) / 24
        )
        self.current_temperature = (
            base_temp + temp_variation + random.uniform(-1.0, 1.0)
        )
        humidity_variation = self.weather_patterns["humidity_range"] * np.sin(
            2 * np.pi * (hour - 12) / 24
        )
        self.current_humidity = max(
            0.2,
            min(1.0, 0.6 + humidity_variation + random.uniform(-0.05, 0.05)),
        )
        wind_index = hour % len(self.weather_patterns["wind_pattern"])
        self.wind_speed = (
            self.weather_patterns["wind_pattern"][wind_index]
            * random.uniform(0.8, 1.2)
        )
        self.solar_irradiance = (
            self.weather_patterns["solar_pattern"][hour]
            * random.uniform(0.7, 1.3)
        )
        self.cloud_cover = random.uniform(0.0, 0.8)
        self.last_update = virtual_state.timestamp

        return {
            "temperature": self.current_temperature,
            "humidity": self.current_humidity,
            "wind_speed": self.wind_speed,
            "solar_irradiance": self.solar_irradiance,
            "cloud_cover": self.cloud_cover,
            "season": self.season,
            "thermal_load_factor": max(
                0.0, 1.0 + (self.current_temperature - 20.0) * 0.05
            ),
        }

    def predict_behavior(
        self,
        horizon_hours: int,
        scenario_data: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Forecast weather indicators with illustrative random variation."""
        predictions = {
            "temperature_forecast": [],
            "humidity_forecast": [],
            "wind_forecast": [],
            "solar_forecast": [],
        }
        current_time = scenario_data.get("current_time", datetime.now())
        base_temp = {"winter": 5.0, "spring": 15.0, "fall": 18.0}.get(
            self.season, 22.0
        )

        for hour_offset in range(1, horizon_hours + 1):
            hour = (current_time + timedelta(hours=hour_offset)).hour
            temp_variation = self.weather_patterns["daily_temp_range"] * np.sin(
                2 * np.pi * (hour - 6) / 24
            )
            predictions["temperature_forecast"].append(
                base_temp + temp_variation + random.uniform(-2.0, 2.0)
            )
            humidity_variation = self.weather_patterns["humidity_range"] * np.sin(
                2 * np.pi * (hour - 12) / 24
            )
            predictions["humidity_forecast"].append(
                max(0.2, min(1.0, 0.6 + humidity_variation + random.uniform(-0.1, 0.1)))
            )
            wind_index = hour % len(self.weather_patterns["wind_pattern"])
            predictions["wind_forecast"].append(
                self.weather_patterns["wind_pattern"][wind_index]
                * random.uniform(0.7, 1.3)
            )
            predictions["solar_forecast"].append(
                self.weather_patterns["solar_pattern"][hour]
                * random.uniform(0.5, 1.5)
            )

        return predictions


@dataclass
class StorageSystemModel(SubsystemModel):
    """Battery model with signed power in kW and energy in kWh."""

    def __init__(self):
        super().__init__("Storage System", "Battery Storage", 15)
        self.capacity = 2000.0  # kWh
        self.max_charge_rate = 500.0  # kW
        self.max_discharge_rate = 500.0  # kW
        self.round_trip_efficiency = 0.92
        self.current_soc = 0.5
        self.charge_efficiency = np.sqrt(self.round_trip_efficiency)
        self.discharge_efficiency = np.sqrt(self.round_trip_efficiency)
        self.current_power = 0.0  # Positive charges; negative discharges, kW
        self.total_energy_charged = 0.0  # kWh added to stored energy
        self.total_energy_discharged = 0.0  # kWh removed from stored energy
        self.cycle_count = 0.0
        self.control_mode = "auto"
        self.price_threshold_charge = 0.10
        self.price_threshold_discharge = 0.18

    def update_state(
        self,
        physical_data: Dict[str, Any],
        virtual_state: DigitalTwinState,
    ) -> Dict[str, Any]:
        """Choose a simple price- or demand-based action for one 15-minute step."""
        interval_hours = 15.0 / 60.0
        current_price = virtual_state.current_price
        current_demand = virtual_state.current_demand
        self.current_power = 0.0

        if self.control_mode == "auto":
            stored_energy = self.current_soc * self.capacity
            if current_price < self.price_threshold_charge and self.current_soc < 0.9:
                energy_room = max(0.0, 0.9 * self.capacity - stored_energy)
                feasible_power = energy_room / (
                    self.charge_efficiency * interval_hours
                )
                self.current_power = min(self.max_charge_rate, feasible_power)
            elif current_price > self.price_threshold_discharge and self.current_soc > 0.1:
                available_energy = max(0.0, stored_energy - 0.1 * self.capacity)
                feasible_power = (
                    available_energy * self.discharge_efficiency / interval_hours
                )
                self.current_power = -min(
                    self.max_discharge_rate, feasible_power
                )
        elif self.control_mode == "peak_shaving":
            if current_demand > 30.0 and self.current_soc > 0.2:
                stored_energy = self.current_soc * self.capacity
                available_energy = max(0.0, stored_energy - 0.1 * self.capacity)
                feasible_power = (
                    available_energy * self.discharge_efficiency / interval_hours
                )
                self.current_power = -min(
                    self.max_discharge_rate,
                    current_demand - 30.0,
                    feasible_power,
                )

        if self.current_power > 0.0:
            energy_added = (
                self.current_power * self.charge_efficiency * interval_hours
            )
            self.current_soc = min(
                0.9, self.current_soc + energy_added / self.capacity
            )
            self.total_energy_charged += energy_added
        elif self.current_power < 0.0:
            energy_removed = (
                abs(self.current_power) / self.discharge_efficiency * interval_hours
            )
            self.current_soc = max(
                0.1, self.current_soc - energy_removed / self.capacity
            )
            self.total_energy_discharged += energy_removed

        self.cycle_count += (
            abs(self.current_power) * interval_hours / (2.0 * self.capacity)
        )
        self.last_update = virtual_state.timestamp

        return {
            "current_soc": self.current_soc,
            "current_power": self.current_power,
            "available_capacity": self.capacity * (1.0 - self.current_soc),
            "available_energy": self.capacity * self.current_soc,
            "total_energy_charged": self.total_energy_charged,
            "total_energy_discharged": self.total_energy_discharged,
            "cycle_count": self.cycle_count,
            "efficiency": self.round_trip_efficiency,
        }

    def predict_behavior(
        self,
        horizon_hours: int,
        scenario_data: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Forecast hourly storage actions while respecting the SOC bounds."""
        predictions = {
            "soc_forecast": [],
            "power_forecast": [],
            "energy_throughput_forecast": [],
        }
        current_soc = self.current_soc
        price_forecast = scenario_data.get(
            "price_forecast", [0.15] * horizon_hours
        )

        for hour_offset in range(horizon_hours):
            future_price = (
                price_forecast[hour_offset]
                if hour_offset < len(price_forecast)
                else 0.15
            )
            predicted_power = 0.0
            if future_price < self.price_threshold_charge and current_soc < 0.9:
                energy_room = (0.9 - current_soc) * self.capacity
                predicted_power = min(
                    self.max_charge_rate,
                    energy_room / self.charge_efficiency,
                )
                current_soc += (
                    predicted_power * self.charge_efficiency / self.capacity
                )
            elif future_price > self.price_threshold_discharge and current_soc > 0.1:
                available_energy = (current_soc - 0.1) * self.capacity
                predicted_power = -min(
                    self.max_discharge_rate,
                    available_energy * self.discharge_efficiency,
                )
                current_soc -= (
                    abs(predicted_power)
                    / (self.discharge_efficiency * self.capacity)
                )

            predictions["soc_forecast"].append(current_soc)
            predictions["power_forecast"].append(predicted_power)
            predictions["energy_throughput_forecast"].append(abs(predicted_power))

        return predictions


weather_system = WeatherSystemModel()
storage_system = StorageSystemModel()

In [ ]:
class IntegratedDigitalTwin:
    """Coordinate subsystem updates, forecasts, and shared virtual state."""

    def __init__(self):
        self.subsystems = {
            "electrical_grid": electrical_grid,
            "thermal_system": thermal_system,
            "equipment_fleet": equipment_fleet,
            "weather_system": weather_system,
            "storage_system": storage_system,
        }
        self.current_state = DigitalTwinState(timestamp=datetime.now())
        self.state_history = deque(maxlen=1000)
        self.sync_frequency = 30
        self.last_sync_time = datetime.now()
        self.sync_accuracy_history = deque(maxlen=100)
        self.prediction_horizon = 24
        self.predictions = {}
        self.prediction_accuracy_history = deque(maxlen=100)
        self.kpi_history = defaultdict(list)
        self.performance_metrics = {
            "energy_cost_savings": 0.0,
            "peak_reduction": 0.0,
            "efficiency_improvement": 0.0,
            "system_availability": 1.0,
        }
        self.scenario_results = {}
        self.current_scenario = "baseline"
        self.optimization_active = False
        self.optimization_recommendations = []

    def synchronize_with_physical_system(
        self,
        physical_data: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Update all subsystems and record one synchronized virtual snapshot."""
        sync_start_time = time.time()
        self.current_state.timestamp = datetime.now()
        subsystem_updates = {}
        errors = {}

        for subsystem_name, subsystem in self.subsystems.items():
            try:
                subsystem_updates[subsystem_name] = subsystem.update_state(
                    physical_data,
                    self.current_state,
                )
            except Exception as error:
                errors[subsystem_name] = str(error)
                subsystem_updates[subsystem_name] = {"error": str(error)}

        # This is a model-confidence proxy, not an observed physical residual.
        confidence_scores = [
            subsystem.accuracy_score
            for name, subsystem in self.subsystems.items()
            if name not in errors
        ]
        model_confidence = (
            float(np.mean(confidence_scores)) if confidence_scores else 0.0
        )
        self.current_state.physical_sync_accuracy = model_confidence
        self.current_state.virtual_sync_accuracy = model_confidence
        self._update_aggregated_state(subsystem_updates)
        self.current_state.system_health_score = float(np.mean([
            1.0 if subsystem.health_status == "healthy" else 0.7
            if subsystem.health_status == "stressed" else 0.3
            for subsystem in self.subsystems.values()
        ]))

        self.state_history.append(deepcopy(self.current_state))
        self.sync_accuracy_history.append(model_confidence)
        self.last_sync_time = self.current_state.timestamp
        sync_duration = time.time() - sync_start_time

        return {
            "sync_success": not errors,
            "sync_accuracy": model_confidence,
            "sync_duration": sync_duration,
            "subsystem_updates": subsystem_updates,
            "errors": errors,
            "timestamp": self.current_state.timestamp,
        }

    def _update_aggregated_state(
        self,
        subsystem_updates: Dict[str, Any],
    ) -> None:
        """Aggregate subsystem outputs without carrying demand across time steps."""
        grid_update = subsystem_updates.get("electrical_grid", {})
        if "current_price" in grid_update:
            self.current_state.current_price = grid_update["current_price"]
        frequency_deviation = grid_update.get("frequency_deviation", 0.0)
        self.current_state.grid_stability_score = max(
            0.0, min(1.0, 1.0 - abs(frequency_deviation) / 2.0)
        )

        thermal_update = subsystem_updates.get("thermal_system", {})
        equipment_update = subsystem_updates.get("equipment_fleet", {})
        weather_update = subsystem_updates.get("weather_system", {})
        storage_update = subsystem_updates.get("storage_system", {})

        thermal_load_kw = thermal_update.get("thermal_load", 0.0)
        thermal_factor = weather_update.get("thermal_load_factor", 1.0)
        equipment_load_kw = equipment_update.get("fleet_power_consumption", 0.0)
        storage_power_kw = storage_update.get("current_power", 0.0)

        self.current_state.current_demand = (
            thermal_load_kw * thermal_factor + equipment_load_kw + storage_power_kw
        ) / 1000.0
        if "comfort_score" in thermal_update:
            self.current_state.thermal_comfort_score = thermal_update["comfort_score"]
        if "fleet_efficiency" in equipment_update:
            self.current_state.equipment_efficiency = equipment_update[
                "fleet_efficiency"
            ]
        if "current_soc" in storage_update:
            self.current_state.storage_soc = storage_update["current_soc"]

    def run_predictive_analytics(
        self,
        horizon_hours: int = 24,
    ) -> Dict[str, Any]:
        """Generate subsystem forecasts and aggregate demand and price forecasts."""
        if horizon_hours < 1:
            raise ValueError("The forecast horizon must be at least one hour.")

        scenario_data = {
            "current_time": self.current_state.timestamp,
            "base_demand": self.current_state.current_demand,
        }
        all_predictions = {}

        for subsystem_name, subsystem in self.subsystems.items():
            subsystem_scenario = dict(scenario_data)
            if subsystem_name == "storage_system":
                subsystem_scenario["price_forecast"] = all_predictions.get(
                    "electrical_grid", {}
                ).get("price_forecast", [])
            try:
                all_predictions[subsystem_name] = subsystem.predict_behavior(
                    horizon_hours,
                    subsystem_scenario,
                )
            except Exception as error:
                all_predictions[subsystem_name] = {"error": str(error)}

        self.predictions = all_predictions
        aggregated_predictions = self._aggregate_predictions(
            all_predictions,
            horizon_hours,
        )
        for state_name in (
            "demand_forecast_1h",
            "demand_forecast_4h",
            "demand_forecast_24h",
            "price_forecast_1h",
            "price_forecast_4h",
            "price_forecast_24h",
        ):
            if state_name in aggregated_predictions:
                setattr(
                    self.current_state,
                    state_name,
                    aggregated_predictions[state_name],
                )

        return {
            "predictions": all_predictions,
            "aggregated_predictions": aggregated_predictions,
            "horizon_hours": horizon_hours,
            "timestamp": self.current_state.timestamp,
        }

    def _aggregate_predictions(
        self,
        all_predictions: Dict[str, Any],
        horizon_hours: int,
    ) -> Dict[str, Any]:
        """Sum compatible subsystem forecasts and calculate horizon summaries."""
        aggregated = {
            "total_demand_forecast": [],
            "price_forecast": [],
            "system_efficiency_forecast": [],
        }
        thermal = all_predictions.get("thermal_system", {}).get(
            "thermal_load_forecast", []
        )
        fleet = all_predictions.get("equipment_fleet", {}).get(
            "power_consumption_forecast", []
        )
        storage = all_predictions.get("storage_system", {}).get(
            "power_forecast", []
        )
        weather = all_predictions.get("weather_system", {}).get(
            "temperature_forecast", []
        )

        for index in range(horizon_hours):
            thermal_kw = thermal[index] if index < len(thermal) else 0.0
            fleet_kw = fleet[index] if index < len(fleet) else 0.0
            storage_kw = storage[index] if index < len(storage) else 0.0
            if index < len(weather):
                thermal_factor = max(0.0, 1.0 + (weather[index] - 20.0) * 0.05)
            else:
                thermal_factor = 1.0
            demand_mw = (
                thermal_kw * thermal_factor + fleet_kw + storage_kw
            ) / 1000.0
            aggregated["total_demand_forecast"].append(demand_mw)

        grid_predictions = all_predictions.get("electrical_grid", {})
        aggregated["price_forecast"] = grid_predictions.get(
            "price_forecast", []
        )
        aggregated["system_efficiency_forecast"] = all_predictions.get(
            "equipment_fleet", {}
        ).get("efficiency_forecast", [])

        demand_forecast = aggregated["total_demand_forecast"]
        price_forecast = aggregated["price_forecast"]
        if demand_forecast:
            aggregated["demand_forecast_1h"] = demand_forecast[0]
            aggregated["demand_forecast_4h"] = float(
                np.mean(demand_forecast[:4])
            )
            aggregated["demand_forecast_24h"] = float(
                np.mean(demand_forecast[:24])
            )
        if price_forecast:
            aggregated["price_forecast_1h"] = price_forecast[0]
            aggregated["price_forecast_4h"] = float(
                np.mean(price_forecast[:4])
            )
            aggregated["price_forecast_24h"] = float(
                np.mean(price_forecast[:24])
            )

        return aggregated


digital_twin = IntegratedDigitalTwin()

In [ ]:
def demonstrate_baseline_scenario(
    digital_twin: IntegratedDigitalTwin,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """Run one illustrative synchronization step and a 24-hour forecast."""
    print("=" * 60)
    print("BASELINE SCENARIO DEMONSTRATION")
    print("=" * 60)

    physical_data = {
        "total_demand": 25.0,  # Observed grid load, MW
        "ambient_temp": 22.0,  # Degrees Celsius
        "wind_speed": 5.0,
        "solar_irradiance": 600,
        "equipment_status": "normal",
        "grid_frequency": 50.0,
    }
    sync_result = digital_twin.synchronize_with_physical_system(physical_data)

    print("\nSynchronization step:")
    print(f"  Update successful: {sync_result['sync_success']}")
    print(
        "  Model-confidence proxy: "
        f"{sync_result['sync_accuracy']:.1%}"
    )
    print(f"  Update duration: {sync_result['sync_duration']:.6f} seconds")
    print(
        "  Modeled flexible facility load: "
        f"{digital_twin.current_state.current_demand:.3f} MW"
    )
    print(f"  Observed grid load: {electrical_grid.current_load:.2f} MW")
    print(f"  Current price: ${digital_twin.current_state.current_price:.3f}/kWh")
    print(f"  Storage state of charge: {digital_twin.current_state.storage_soc:.1%}")

    print("\nRunning predictive analytics for a 24-hour horizon...")
    predictions = digital_twin.run_predictive_analytics(horizon_hours=24)
    aggregated = predictions["aggregated_predictions"]
    print("\nSelected forecast summaries:")
    for key, label, unit in (
        ("demand_forecast_1h", "Mean modeled demand (1 h)", "MW"),
        ("demand_forecast_4h", "Mean modeled demand (first 4 h)", "MW"),
        ("price_forecast_1h", "Price forecast (1 h)", "$/kWh"),
        ("price_forecast_4h", "Mean price forecast (first 4 h)", "$/kWh"),
    ):
        if key in aggregated:
            print(f"  {label}: {aggregated[key]:.3f} {unit}")

    return sync_result, predictions


baseline_results, baseline_predictions = demonstrate_baseline_scenario(digital_twin)

In [ ]:
def create_digital_twin_visualizations(
    sync_result: Dict[str, Any],
    predictions: Dict[str, Any],
    digital_twin: IntegratedDigitalTwin,
) -> None:
    """Plot current indicators, forecasts, subsystem health, and KPIs."""
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(18, 11),
        subplot_kw={"polar": False},
    )
    fig.suptitle(
        "Integrated Digital-Twin Energy-System Simulation",
        fontsize=16,
        fontweight="bold",
    )

    # Values have different units, so each bar is labelled with its unit.
    ax = axes[0, 0]
    metrics = ["Demand", "Price", "Storage SOC", "Efficiency"]
    values = [
        digital_twin.current_state.current_demand,
        digital_twin.current_state.current_price * 100,
        digital_twin.current_state.storage_soc * 100,
        digital_twin.current_state.equipment_efficiency * 100,
    ]
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
    bars = ax.bar(metrics, values, color=colors, alpha=0.75)
    ax.set_ylabel("Displayed value (units shown above bars)")
    ax.set_title("Current Virtual-System Indicators")
    ax.grid(axis="y", alpha=0.3)
    labels = [
        f"{values[0]:.2f} MW",
        f"${digital_twin.current_state.current_price:.3f}/kWh",
        f"{values[2]:.1f}%",
        f"{values[3]:.1f}%",
    ]
    for bar, label in zip(bars, labels):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            label,
            ha="center",
            va="bottom",
            fontsize=8,
        )
    ax.tick_params(axis="x", rotation=20)

    aggregated = predictions["aggregated_predictions"]
    hours = list(range(1, len(aggregated["total_demand_forecast"]) + 1))

    ax = axes[0, 1]
    demand_forecast = aggregated["total_demand_forecast"]
    if demand_forecast:
        ax.plot(hours, demand_forecast, color="tab:blue", marker="o", markersize=3, label="Modeled forecast")
        ax.fill_between(
            hours,
            [value * 0.9 for value in demand_forecast],
            [value * 1.1 for value in demand_forecast],
            color="tab:blue",
            alpha=0.2,
            label="Illustrative ±10% band",
        )
        ax.axhline(
            digital_twin.current_state.current_demand,
            color="tab:red",
            linestyle="--",
            label="Current modeled load",
        )
    ax.set_xlabel("Forecast hour")
    ax.set_ylabel("Modeled flexible load (MW)")
    ax.set_title("Modeled Demand Forecast")
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[0, 2]
    price_forecast = aggregated["price_forecast"]
    if price_forecast:
        price_hours = list(range(1, len(price_forecast) + 1))
        ax.plot(price_hours, price_forecast, color="tab:orange", marker="s", markersize=3, label="Forecast")
        ax.axhline(
            digital_twin.current_state.current_price,
            color="tab:red",
            linestyle="--",
            label="Current price",
        )
    ax.set_xlabel("Forecast hour")
    ax.set_ylabel("Price (dollars per kWh)")
    ax.set_title("Grid Price Forecast")
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[1, 0]
    subsystem_names = list(digital_twin.subsystems)
    health_scores = [
        1.0 if digital_twin.subsystems[name].health_status == "healthy" else 0.7
        if digital_twin.subsystems[name].health_status == "stressed" else 0.3
        for name in subsystem_names
    ]
    health_colors = [
        "tab:green" if score >= 0.8 else "tab:orange" if score >= 0.6 else "tab:red"
        for score in health_scores
    ]
    bars = ax.bar(subsystem_names, health_scores, color=health_colors, alpha=0.75)
    ax.set_ylabel("Illustrative health score")
    ax.set_title("Subsystem Health Indicators")
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.3)
    for bar, score in zip(bars, health_scores):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{score:.0%}",
            ha="center",
            fontsize=8,
        )

    ax = axes[1, 1]
    sync_history = list(digital_twin.sync_accuracy_history)
    if sync_history:
        ax.plot(
            range(1, len(sync_history) + 1),
            sync_history,
            color="tab:green",
            marker="o",
            label="Model-confidence proxy",
        )
    ax.set_xlabel("Recorded synchronization step")
    ax.set_ylabel("Subsystem model-confidence proxy")
    ax.set_title("Recorded Update Confidence")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=0.3)

    # A polar subplot displays the five normalized indicators on a common scale.
    ax = fig.add_subplot(2, 3, 6, projection="polar")
    categories = ["Efficiency", "Comfort", "Health", "Model confidence", "Stability"]
    kpi_values = [
        digital_twin.current_state.equipment_efficiency,
        digital_twin.current_state.thermal_comfort_score,
        digital_twin.current_state.system_health_score,
        sync_result["sync_accuracy"],
        digital_twin.current_state.grid_stability_score,
    ]
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    closed_angles = angles + angles[:1]
    closed_values = kpi_values + kpi_values[:1]
    ax.plot(closed_angles, closed_values, "o-", linewidth=2, color="tab:blue")
    ax.fill(closed_angles, closed_values, alpha=0.2, color="tab:blue")
    ax.set_xticks(angles)
    ax.set_xticklabels(categories, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title("Normalized System Indicators", pad=20)
    ax.grid(True)

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    overall_performance = float(np.mean(kpi_values))
    if overall_performance >= 0.90:
        status = "High on the selected illustrative indicators"
    elif overall_performance >= 0.80:
        status = "Moderate-to-high on the selected illustrative indicators"
    elif overall_performance >= 0.70:
        status = "Moderate on the selected illustrative indicators"
    else:
        status = "Low on the selected illustrative indicators"

    print("\nDigital-Twin Simulation Summary")
    print(f"  Subsystems represented: {len(digital_twin.subsystems)}")
    print(f"  Forecast horizon: {predictions['horizon_hours']} hours")
    print(f"  Intended synchronization interval: {digital_twin.sync_frequency} seconds")
    print(f"  Modeled flexible facility load: {digital_twin.current_state.current_demand:.3f} MW")
    print(f"  Observed grid load: {electrical_grid.current_load:.2f} MW")
    print(f"  Grid price: ${digital_twin.current_state.current_price:.3f}/kWh")
    print(f"  Storage state of charge: {digital_twin.current_state.storage_soc:.1%}")
    print(f"  Mean of selected normalized indicators: {overall_performance:.1%}")
    print(f"  Indicator summary: {status}")
    print("  The confidence value is a model proxy, not measured synchronization accuracy.")


create_digital_twin_visualizations(
    baseline_results,
    baseline_predictions,
    digital_twin,
)

## Interpreting the Integrated Simulation

The subsystem models combine several kinds of information in one virtual state. The electrical-grid model uses observed grid demand to estimate a load ratio and generate a price indicator. The thermal and equipment models estimate power in kW. The weather model supplies an illustrative factor for the thermal contribution, and the storage model contributes signed power according to its simple price-based or peak-shaving rule. The integration step converts the resulting power to MW before recording modeled flexible facility demand.

The forecasts are scenario estimates, not validated predictions. Their values depend on the simplified patterns and seeded random variation in the model. The shaded demand band is a visual illustration of a ±10% range; it is not inferred from historical forecast errors. The model-confidence proxy is based on subsystem configuration scores and should not be interpreted as evidence that a physical system is synchronized to a particular accuracy.

### Strengths and limitations

**Strengths**

- A shared state makes the relationship between subsystem updates and system-level indicators visible.
- Forecasts are generated by the subsystem models and aggregated using explicit power-unit conversions.
- The storage model uses a consistent sign convention: charging increases modeled demand, while discharging decreases it.
- The fixed random seeds make the illustrative random variations reproducible in a given software environment.

**Limitations**

- The supplied observations are synthetic, and subsystem behavior is simplified rather than calibrated against facility data.
- The simulation does not implement real-time bidirectional communication, measured synchronization error, a 15-minute forecast resolution, or a validated predictive-accuracy assessment.
- The notebook does not execute what-if disruption scenarios, calculate energy-cost savings, or optimize equipment schedules.
- Grid voltage and frequency dynamics, storage degradation, thermal dynamics, and operational constraints are represented only partially.

### Further study

A natural extension is to provide measured observations and compare model forecasts with later observations, while retaining the same subsystem-based simulation structure. Additional scenarios could represent specified weather or equipment conditions. Those extensions require suitable input data and validation before their outputs can support operational decisions.